# Arabic Grammatical Error Correction (GEC) Notebook

This notebook demonstrates a pipeline for Arabic Grammatical Error Correction (GEC), leveraging various NLP tools and a fine-tuned Gemma 3 model.

## 1. Setup and Dependencies

This section installs the necessary open-source libraries and downloads essential data for Arabic NLP tasks. It includes `transformers` for the GEC model, `camel-tools` for morphological analysis, `langdetect` for language identification, `pyarabic` and `tashaphyne` for basic Arabic text processing, and `symspellpy` for spelling correction.

In [1]:
# Install the required open-source libraries
!pip install transformers torch
!pip install langdetect
!pip install pyarabic camel-tools
!pip install qalsadi nltk

# Core Arabic NLP toolkit (morphology, disambiguation, lemmatization, reinflection)
# + spelling (symspellpy), stemming (tashaphyne, pulls in pyarabic), language ID (langdetect)
!pip install -q camel-tools symspellpy tashaphyne langdetect pandas

import sys
print(f"Python: {sys.version.split()[0]}")
if sys.version_info < (3, 11):
    print("camel-tools needs Python 3.11+. In Colab: Runtime > Change runtime type.")

!camel_data -i morphology-db-msa-r13
!camel_data -i disambig-mle-calima-msa-r13

!wget -q -O ar_50k.txt "https://raw.githubusercontent.com/hermitdave/FrequencyWords/master/content/2016/ar/ar_50k.txt"
!wc -l ar_50k.txt

# Download NLTK tokenization data
import nltk
nltk.download('punkt')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 31.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=7669f952b3570700a99eecfbc852636980e1540c5a3fee58de9e691a0b909e4a
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.7/125.7 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.0/120.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.3/122.3 kB 4.9 M

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

## 2. Initialize Arabic NLP Tools

Here, we initialize several key components for Arabic natural language processing from the `camel_tools` library, along with `symspellpy` for robust spelling correction. These tools are crucial for analyzing, disambiguating, and correcting Arabic text.

- **Morphological Analyzer (`analyzer`):** Used to break down Arabic words into their constituent morphemes (e.g., prefixes, stems, suffixes) and identify their root, part-of-speech, and other features.
- **Morphological Generator (`generator`):** Allows for generating different forms of Arabic words based on their root and morphological features.
- **MLE Disambiguator (`mle_disambiguator`):** Resolves ambiguities in Arabic words by predicting the most likely part-of-speech, gender, number, and lemma.
- **Spelling Corrector (`sym_spell`):** Utilizes a frequency dictionary (`ar_50k.txt`) to suggest corrections for misspelled words based on edit distance.
- **Light Stemmer (`light_stemmer`):** Provides a simpler form of stemming for Arabic words.

In [2]:
import re
import html
import pandas as pd
from IPython.display import display, HTML

from camel_tools.tokenizers.word import simple_word_tokenize
from camel_tools.morphology.database import MorphologyDB
from camel_tools.morphology.analyzer import Analyzer
from camel_tools.morphology.generator import Generator
from camel_tools.disambig.mle import MLEDisambiguator
from camel_tools.utils.dediac import dediac_ar
from camel_tools.utils.normalize import (
    normalize_alef_ar, normalize_alef_maksura_ar, normalize_teh_marbuta_ar,
)
import pyarabic.araby as araby
from tashaphyne.stemming import ArabicLightStemmer
from symspellpy import SymSpell, Verbosity
from langdetect import detect_langs, DetectorFactory, LangDetectException
DetectorFactory.seed = 0  # deterministic langdetect results

print("Loading morphological analyzer ...")
analyzer = Analyzer(MorphologyDB.builtin_db())

print("Loading morphological generator ...")
generator = Generator(MorphologyDB.builtin_db(flags='g'))

print("Loading MLE disambiguator (POS / gender / number / lemma) ...")
mle_disambiguator = MLEDisambiguator.pretrained()

print("Loading spelling frequency dictionary ...")
sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)
sym_spell.load_dictionary('ar_50k.txt', term_index=0, count_index=1, separator=' ', encoding='utf-8')

light_stemmer = ArabicLightStemmer()

print("Ready.")

Loading morphological analyzer ...
Loading morphological generator ...
Loading MLE disambiguator (POS / gender / number / lemma) ...
Loading spelling frequency dictionary ...
Ready.


## 3. Input Text and Language Verification

We define the input Arabic text that needs correction. A preliminary step is to verify that the text is indeed Arabic, using `langdetect`, to ensure the subsequent Arabic NLP tools are applied correctly.

In [18]:
from langdetect import detect, LangDetectException

def verify_arabic(text):
    try:
        lang = detect(text)
        if lang == 'ar':
            print("✅ Language verified: Arabic")
            return True
        else:
            print(f"❌ Text is not Arabic. Detected language: {lang}")
            return False
    except LangDetectException:
        print("❌ Could not detect language. The text might be too short or invalid.")
        return False

# Feel free to change this test sentence to test different errors!
# This sentence contains spelling/grammar mistakes: "سعيداً" should be "سعيدٌ", "الي المدرسه" should be "إلى المدرسة", etc.
user_text = "اكلت الغنت النفاحة وهى سعيد"
# user_text = "الغنت اكل النفاحة وهى سغيد"

is_arabic = verify_arabic(user_text)

✅ Language verified: Arabic


## 4. Tokenization

This step tokenizes the input Arabic text into individual words or sub-word units. The `is_arabic_token` function helps filter out non-Arabic tokens, ensuring that only relevant parts of the text are processed by Arabic-specific tools.

In [19]:
ARABIC_LETTERS_RE = re.compile(r'[\u0621-\u063A\u0641-\u064A\u066E\u066F\u0671-\u06D3\u06D5]')
LATIN_RE = re.compile(r'[A-Za-z]')

def is_arabic_token(tok):
    return bool(ARABIC_LETTERS_RE.search(tok)) and not LATIN_RE.search(tok) and not tok.isdigit()

def tokenize_arabic(text):
    return simple_word_tokenize(text)

tokens = tokenize_arabic(user_text)
print(tokens)
print(f"\n{len(tokens)} tokens, {sum(is_arabic_token(t) for t in tokens)} of which are Arabic words")

['اكلت', 'الغنت', 'النفاحة', 'وهى', 'سعيد']

5 tokens, 5 of which are Arabic words


## 5. Detect Spelling Errors

We identify potential spelling errors by checking if a token has zero morphological analyses according to the `camel_tools` analyzer. If the analyzer cannot find any valid interpretations for a word, it's flagged as a candidate for a spelling mistake.

In [20]:
def detect_spelling_errors(tokens):
    # Returns {token: [positions]} for tokens with zero morphological analyses.
    errors = {}
    for i, tok in enumerate(tokens):
        if not is_arabic_token(tok):
            continue
        if len(analyzer.analyze(tok)) == 0:
            errors.setdefault(tok, []).append(i)
    return errors

spelling_errors = detect_spelling_errors(tokens)

print(f"{len(spelling_errors)} candidate spelling error(s) found:")
for word, positions in spelling_errors.items():
    print(f"  - '{word}'  at position(s) {positions}")

2 candidate spelling error(s) found:
  - 'الغنت'  at position(s) [1]
  - 'النفاحة'  at position(s) [2]


## 6. Correct Spelling Errors (Rule-Based)

This section applies a rule-based spelling correction using `symspellpy`. For each flagged spelling error, it looks up suggestions in the Arabic frequency dictionary (`ar_50k.txt`) and prioritizes suggestions that are morphologically valid according to `camel-tools`.

In [21]:
def correct_spelling(tokens, spelling_errors):
    corrections = {}
    for word in spelling_errors:
        suggestions = sym_spell.lookup(word, Verbosity.CLOSEST, max_edit_distance=2)
        valid = [s.term for s in suggestions if analyzer.analyze(s.term)]
        if valid:
            corrections[word] = valid[0]
        elif suggestions:
            corrections[word] = suggestions[0].term  # no analyzer-valid candidate; best-effort fallback
        else:
            corrections[word] = None  # nothing found; leave the original word as-is

    corrected_tokens = [
        corrections[tok] if (tok in corrections and corrections[tok]) else tok
        for tok in tokens
    ]
    return corrected_tokens, corrections

spelling_corrected_tokens, spelling_corrections = correct_spelling(tokens, spelling_errors)

for original, fixed in spelling_corrections.items():
    print(f"  '{original}'  ->  '{fixed}'")
print("\nText after spelling correction:")
spell_corrected_text = ' '.join(spelling_corrected_tokens)
print(spell_corrected_text)


  'الغنت'  ->  'البنت'
  'النفاحة'  ->  'التفاحة'

Text after spelling correction:
اكلت البنت التفاحة وهى سعيد


## 7. Grammatical Error Correction using Gemma 3

For more complex grammatical corrections beyond simple spelling, we utilize a pre-trained Arabic GEC model based on Gemma 3 (`alnnahwi/gemma-3-1b-arabic-gec-v1`) from Hugging Face Transformers. This model is specifically fine-tuned for Arabic Grammatical Error Correction and can address issues like word order, verb conjugations, and agreement.

In [22]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Note: You must log into Hugging Face using `huggingface-cli login` first
# and accept the Gemma 3 license on their website.
model_name = "alnnahwi/gemma-3-1b-arabic-gec-v1"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

# Wrap the text in the prompt format the model was trained on
messages = [{"role": "user", "content": spell_corrected_text}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=256)

corrected_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("Corrected:", corrected_text)

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Corrected: model
أكلت البنت التفاحة وهي سعيدة
